# Germany Hourly Electricity Generation — Standard Dataset Project

**Questions-based analysis and forecasting**

This notebook uses the original standard-data layout shown in the supplied screenshots:

- Metadata table: one row per `curve_id`, with technology and fuel labels.
- `generation_series.csv`: hourly observations in wide format. The first column is `reference_time_start`; each remaining column is a `curve_id`.

All analysis is built from these two source tables. No simulated data are used.

**Required input files**
1. `generation_series.csv`
2. The metadata CSV containing `curve_id`, `Technology_is`, and `Technology_Fuel_is`.

Set their paths in Section 0. The screenshots show the schema and a time span from 2014-12-01 to 2022-07-13, but do not contain the underlying CSV values. Therefore, numerical findings and model scores will be generated when the actual CSV files are supplied and the notebook is run.

## 0. Imports and file paths

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import STL

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor, RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.base import clone

RANDOM_STATE = 42
DATA_DIR = Path(".")  # change if your CSV files are in another folder
SERIES_PATH = DATA_DIR / "generation_series.csv"
META_PATH = DATA_DIR / "curve_metadata.csv"  # change to your metadata filename

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## 1. Load and understand the standard dataset

In [ ]:
# Read the wide hourly series and the curve metadata.
series = pd.read_csv(SERIES_PATH)
meta = pd.read_csv(META_PATH)

print("Series shape:", series.shape)
print("Metadata shape:", meta.shape)
display(series.head())
display(meta.head())
display(meta.dtypes.to_frame("dtype"))

In [ ]:
# Normalize column names and identify the timestamp column.
series.columns = series.columns.astype(str).str.strip()
meta.columns = meta.columns.astype(str).str.strip()

time_col = "reference_time_start"
required_meta = ["curve_id", "Technology_is", "Technology_Fuel_is"]
missing = [c for c in required_meta if c not in meta.columns]
if missing:
    raise ValueError(f"Metadata is missing columns: {missing}")

if time_col not in series.columns:
    raise ValueError(f"Series file must contain '{time_col}'")

series[time_col] = pd.to_datetime(series[time_col], utc=True, errors="coerce")
series = series.dropna(subset=[time_col]).sort_values(time_col)
series = series.drop_duplicates(time_col, keep="last").set_index(time_col)

# Curve IDs are numeric in the metadata but appear as column names in the wide file.
meta["curve_id"] = meta["curve_id"].astype(str).str.strip()
curve_cols = [c for c in series.columns if c in set(meta["curve_id"])]
if not curve_cols:
    raise ValueError("No metadata curve_id matches columns in generation_series.csv.")

series[curve_cols] = series[curve_cols].apply(pd.to_numeric, errors="coerce")
print("Matched curves:", len(curve_cols), "/", len(meta))
print("Time range:", series.index.min(), "to", series.index.max())
print("Hourly gaps:", len(pd.date_range(series.index.min(), series.index.max(), freq="h").difference(series.index)))

### 1.1 Build a clean curve dictionary and inspect coverage

In [ ]:
meta["Technology_is"] = meta["Technology_is"].astype("string").str.strip()
meta["Technology_Fuel_is"] = meta["Technology_Fuel_is"].astype("string").str.strip()

coverage = []
for _, row in meta.iterrows():
    cid = str(row["curve_id"])
    if cid in series.columns:
        s = series[cid]
        coverage.append({
            "curve_id": cid,
            "technology": row["Technology_is"],
            "fuel": row["Technology_Fuel_is"],
            "non_missing": s.notna().sum(),
            "missing_pct": s.isna().mean() * 100,
            "mean_MW": s.mean()
        })
coverage = pd.DataFrame(coverage).sort_values(["technology", "fuel"])
display(coverage)

### 1.2 Map individual curves to technology groups

Rows labelled `Power generation` are treated as aggregate curves and excluded from source-level sums. Including these alongside individual technologies could double-count generation. The exact technology/fuel mapping is printed above for manual verification.


In [ ]:
# Use the labels in the metadata; inspect unique values before changing mappings.
display(meta[["Technology_is", "Technology_Fuel_is"]].drop_duplicates().sort_values(
    ["Technology_is", "Technology_Fuel_is"]
).to_string(index=False))

def label_text(x):
    return "" if pd.isna(x) else str(x).strip().lower()

m = meta[meta["curve_id"].isin(curve_cols)].copy()
m["tech_text"] = m["Technology_is"].map(label_text)
m["fuel_text"] = m["Technology_Fuel_is"].map(label_text)

# Main source groups. Adjust patterns if the metadata uses different wording.
patterns = {
    "Thermal": ["thermal power"],
    "Wind": ["wind turbines", "wind power"],
    "Solar": ["solar cell", "solar"],
    "Hydro": ["reservoir", "run of river", "hydro", "pump storage"],
    "Nuclear": ["nuclear"],
    "Geothermal": ["geothermal"],
    "Other": ["biomass", "manne power"]
}

def assign_group(tech):
    # Aggregate rows such as "Power generation" may already sum multiple sources.
    # Exclude them from source-level totals to avoid double counting.
    if "power generation" in tech:
        return "Aggregate"
    for group, keys in patterns.items():
        if any(k in tech for k in keys):
            return group
    return "Other"

m["source"] = m["tech_text"].map(assign_group)

# Pump storage is kept separately for Question 3 and excluded from conventional hydro supply.
m.loc[m["tech_text"].str.contains("pump storage", na=False), "source"] = "PumpedStorage"

display(m[["curve_id", "Technology_is", "Technology_Fuel_is", "source"]])

## 2. Question 1 — Hourly electricity supply by source

Aggregate all curves assigned to each source by summing their hourly MW values. Thermal fuel subcategories are therefore combined into total Thermal generation; onshore and offshore wind are combined into Wind. Pumped storage is kept separate because it is a storage technology rather than a primary generation source.

Missing values are not automatically interpreted as zero. The aggregation uses `min_count=1`, so an hour remains missing if every curve in that group is missing.

In [ ]:
source_series = {}
for source in ["Thermal", "Wind", "Solar", "Hydro", "Nuclear", "Geothermal", "Other", "PumpedStorage"]:
    ids = m.loc[m["source"] == source, "curve_id"].tolist()
    ids = [c for c in ids if c in series.columns]
    if ids:
        source_series[source] = series[ids].sum(axis=1, min_count=1)

hourly = pd.DataFrame(source_series).sort_index()
display(hourly.head())
display(hourly.isna().mean().mul(100).round(2).rename("missing_pct").to_frame())

In [ ]:
fig, axes = plt.subplots(len(hourly.columns), 1, figsize=(15, 2.5 * len(hourly.columns)), sharex=True)
axes = np.atleast_1d(axes)
for ax, col in zip(axes, hourly.columns):
    ax.plot(hourly.index, hourly[col], lw=0.45, alpha=0.65, label="Hourly")
    ax.plot(hourly.index, hourly[col].rolling(24*7, min_periods=24).mean(), lw=1.2, label="7-day moving average")
    ax.set_title(f"{col} generation (MW)")
    ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics and data-quality diagnostics.
summary = hourly.agg(["count", "mean", "std", "min", "median", "max"]).T
summary["cv"] = summary["std"] / summary["mean"].abs()
display(summary.sort_values("mean", ascending=False))

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(hourly.isna().resample("MS").mean().T, cmap="Reds", cbar_kws={"label": "Missing share"}, ax=ax)
ax.set_title("Monthly missing-data share by source")
ax.set_xlabel("Month")
plt.tight_layout()
plt.show()

**Discussion guide:** describe long-run level changes from the moving averages; daily and weekly repetition from recurring peaks/troughs; annual seasonality from winter/summer patterns; and data issues from missingness, gaps, abrupt discontinuities, and unusually large spikes. Do not claim a trend or seasonal pattern until the plots from the actual CSVs support it.

## 3. Question 2 — Daily Solar generation, seasonality and distribution

In [ ]:
if "Solar" not in hourly:
    raise ValueError("No Solar curves were matched. Check Technology_is mapping in Section 1.2.")

solar_hourly = hourly["Solar"]
solar_daily = solar_hourly.resample("D").sum(min_count=1)

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(solar_daily.index, solar_daily, lw=0.7, alpha=0.7, label="Daily generation")
ax.plot(solar_daily.index, solar_daily.rolling(30, min_periods=15).mean(), lw=1.5, label="30-day moving average")
ax.set_title("Daily Solar generation (MWh)")
ax.set_ylabel("MWh")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(solar_daily.dropna(), bins=50, kde=True, ax=ax)
ax.set_title("Daily Solar generation — raw distribution")
plt.tight_layout()
plt.show()

In [ ]:
# STL with a yearly period (365 daily observations); robust=True reduces outlier influence.
solar_for_stl = solar_daily.asfreq("D").interpolate("time", limit=7, limit_area="inside")
solar_stl = STL(solar_for_stl, period=365, robust=True).fit()
solar_adjusted = solar_for_stl - solar_stl.seasonal

fig = solar_stl.plot()
fig.set_size_inches(14, 8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(solar_adjusted.dropna(), bins=50, kde=True, ax=ax)
ax.set_title("Daily Solar generation — after removing annual seasonality")
plt.tight_layout()
plt.show()

print("Raw skew:", solar_daily.skew(), "Raw kurtosis:", solar_daily.kurt())
print("Adjusted skew:", solar_adjusted.skew(), "Adjusted kurtosis:", solar_adjusted.kurt())

In [ ]:
# Normality diagnostics: Q-Q plot and tests. Large samples make tests very sensitive.
from statsmodels.graphics.gofplots import qqplot
from scipy.stats import normaltest, jarque_bera

fig, ax = plt.subplots(figsize=(6, 6))
qqplot(solar_adjusted.dropna(), line="s", ax=ax)
ax.set_title("Q-Q plot: seasonally adjusted daily Solar")
plt.tight_layout()
plt.show()

z = solar_adjusted.dropna()
if len(z) >= 8:
    print("D'Agostino K²:", normaltest(z))
    print("Jarque-Bera:", jarque_bera(z))

# Further transformations: Yeo-Johnson handles zero and negative values.
from sklearn.preprocessing import PowerTransformer
z2 = z.to_numpy().reshape(-1, 1)
yj = PowerTransformer(method="yeo-johnson", standardize=False).fit_transform(z2).ravel()
print("Yeo-Johnson skew:", pd.Series(yj).skew())
print("Yeo-Johnson kurtosis:", pd.Series(yj).kurt())
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(yj, bins=50, kde=True, ax=ax)
ax.set_title("Seasonally adjusted Solar — Yeo-Johnson")
plt.tight_layout()
plt.show()

**Discussion guide:** daily aggregation removes intraday variation but retains annual seasonality. STL's seasonal component is subtracted before inspecting the adjusted histogram. Discuss whether the adjusted distribution is more symmetric, whether tails remain heavy, and whether a Yeo–Johnson transform improves skewness. Normality is not guaranteed or required for forecasting with tree-based models.

## 4. Question 3 — Pumped storage hourly mean and shape

In [ ]:
pump_ids = m.loc[m["source"] == "PumpedStorage", "curve_id"].tolist()
if not pump_ids:
    raise ValueError("No Pump storage curve found. Check the metadata Technology_is labels.")

pump_hourly = series[pump_ids].sum(axis=1, min_count=1)
pump_by_hour = pump_hourly.groupby(pump_hourly.index.hour).mean()

display(pump_by_hour.rename("mean_MW").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pump_by_hour.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Mean Pumped Storage by hour of day")
axes[0].set_xlabel("Hour (UTC)")
axes[0].set_ylabel("Mean MW")
sns.histplot(pump_hourly.dropna(), bins=60, kde=True, ax=axes[1])
axes[1].set_title("Hourly Pumped Storage distribution")
plt.tight_layout()
plt.show()

print("Hourly mean:", pump_hourly.mean())
print("Hourly median:", pump_hourly.median())
print("Skewness:", pump_hourly.skew())
print("Share zero:", (pump_hourly == 0).mean())

**Interpretation:** the hourly mean profile shows when the plant tends to have higher or lower net generation. If the series records net power, pumping may appear as negative values and generation as positive values; if the source records generation only, pumping may be absent or represented differently. Confirm the sign convention from the data documentation before interpreting the shape as buy-low / generate-high behavior.

## 5. Part 2 — Predict hourly Thermal generation using Wind and Solar

At time t, Wind and Solar generation are assumed known, while Thermal generation is unknown. The target is total Thermal generation, aggregated across all thermal fuel curves. Only Wind and Solar series are used as predictors; no Thermal lags or other source series are included.

In [ ]:
required = ["Thermal", "Wind", "Solar"]
missing = [c for c in required if c not in hourly.columns]
if missing:
    raise ValueError(f"Missing required source series: {missing}")

model_df = hourly[required].copy()
model_df = model_df.dropna(subset=["Thermal", "Wind", "Solar"])

# Build features using current-time Wind/Solar only plus their past-only dynamics.
def make_features(data):
    X = pd.DataFrame(index=data.index)
    for col in ["Wind", "Solar"]:
        s = data[col]
        X[f"{col}_t"] = s
        X[f"{col}_lag1"] = s.shift(1)
        X[f"{col}_lag24"] = s.shift(24)
        X[f"{col}_lag168"] = s.shift(168)
        X[f"{col}_change1"] = s.diff(1)
        X[f"{col}_change24"] = s.diff(24)
        X[f"{col}_vol24"] = s.diff().rolling(24).std()
        X[f"{col}_mean24"] = s.rolling(24).mean()
        X[f"{col}_mean168"] = s.rolling(168).mean()
        X[f"{col}_shock168"] = (s - s.rolling(168).mean()) / (s.rolling(168).std() + 1e-6)

    total = data["Wind"] + data["Solar"]
    X["Renewable_total_t"] = total
    X["Wind_share_t"] = data["Wind"] / (total + 1)
    X["Solar_share_t"] = data["Solar"] / (total + 1)
    X["Wind_Solar_diff"] = data["Wind"] - data["Solar"]
    X["Renewable_change24"] = total.diff(24)

    idx = data.index
    X["hour_sin"] = np.sin(2*np.pi*idx.hour/24)
    X["hour_cos"] = np.cos(2*np.pi*idx.hour/24)
    X["weekday_sin"] = np.sin(2*np.pi*idx.dayofweek/7)
    X["weekday_cos"] = np.cos(2*np.pi*idx.dayofweek/7)
    X["month_sin"] = np.sin(2*np.pi*idx.month/12)
    X["month_cos"] = np.cos(2*np.pi*idx.month/12)
    return X.replace([np.inf, -np.inf], np.nan)

X_all = make_features(model_df[["Wind", "Solar"]])
y_all = model_df["Thermal"]

# Keep chronological order. First 70% train, next 15% validation, last 15% test.
n = len(model_df)
i_train = int(n * 0.70)
i_val = int(n * 0.85)
train_idx = model_df.index[:i_train]
val_idx = model_df.index[i_train:i_val]
test_idx = model_df.index[i_val:]

X_train, X_val, X_test = X_all.loc[train_idx], X_all.loc[val_idx], X_all.loc[test_idx]
y_train, y_val, y_test = y_all.loc[train_idx], y_all.loc[val_idx], y_all.loc[test_idx]

# Remove rows with unavailable lag/rolling features, keeping each split aligned.
def align_xy(X, y):
    ok = X.notna().all(axis=1) & y.notna()
    return X.loc[ok], y.loc[ok]

X_train, y_train = align_xy(X_train, y_train)
X_val, y_val = align_xy(X_val, y_val)
X_test, y_test = align_xy(X_test, y_test)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
print("Train:", y_train.index.min(), "to", y_train.index.max())
print("Validation:", y_val.index.min(), "to", y_val.index.max())
print("Test:", y_test.index.min(), "to", y_test.index.max())

In [ ]:
# Baseline: current Wind + Solar only, using Ridge.
base_cols = ["Wind_t", "Solar_t"]
baseline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10))
baseline.fit(X_train[base_cols], y_train)
base_pred = baseline.predict(X_test[base_cols])

def score(y, pred):
    return {
        "RMSE": mean_squared_error(y, pred) ** 0.5,
        "MAE": mean_absolute_error(y, pred),
        "R2": r2_score(y, pred)
    }

print("Ridge baseline:", score(y_test, base_pred))

In [ ]:
# Compare representative CPU models on the same feature set.
models = {
    "Ridge": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10)),
    "RandomForest": make_pipeline(SimpleImputer(strategy="median"),
        RandomForestRegressor(n_estimators=250, min_samples_leaf=3, max_features=0.8,
                              n_jobs=-1, random_state=RANDOM_STATE)),
    "ExtraTrees": make_pipeline(SimpleImputer(strategy="median"),
        ExtraTreesRegressor(n_estimators=300, min_samples_leaf=3, max_features=0.8,
                            n_jobs=-1, random_state=RANDOM_STATE)),
    "GradientBoosting": make_pipeline(SimpleImputer(strategy="median"),
        GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=2,
                                  loss="huber", random_state=RANDOM_STATE)),
    "HistGradientBoosting": make_pipeline(SimpleImputer(strategy="median"),
        HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05,
                                      max_leaf_nodes=20, l2_regularization=1,
                                      random_state=RANDOM_STATE))
}

val_rows, fitted = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    val_rows.append({"model": name, **score(y_val, pred)})
    fitted[name] = model

val_results = pd.DataFrame(val_rows).sort_values("RMSE").reset_index(drop=True)
display(val_results)

In [ ]:
# Select using validation only; evaluate the selected model once on Test.
best_name = val_results.loc[0, "model"]
best_model = fitted[best_name]
test_pred = best_model.predict(X_test)

print("Selected model:", best_name)
display(pd.DataFrame([score(y_test, test_pred)], index=[best_name]))

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(y_test.index, y_test, label="Actual Thermal", lw=0.8)
ax.plot(y_test.index, test_pred, label="Predicted Thermal", lw=0.8)
ax.set_title("Thermal generation: actual vs predicted")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

residuals = y_test - test_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(residuals, bins=50, kde=True, ax=axes[0])
axes[0].set_title("Test residual distribution")
plot_acf(residuals.dropna(), lags=48, ax=axes[1])
axes[1].set_title("Test residual ACF")
plt.tight_layout()
plt.show()

### Model discussion checklist

Discuss (i) why Wind and Solar are economically relevant predictors under the must-run / zero marginal cost assumption; (ii) whether their contemporaneous relationship with Thermal is negative; (iii) whether daily/weekly and seasonal effects remain in residuals; (iv) how much the model improves over the Ridge current-generation baseline; and (v) limitations such as demand, imports/exports, outages, prices, and other generation sources being omitted. Do not claim causality: this is a predictive model.

## 6. Reproducibility and final checks

In [ ]:
# Check that no target-derived columns entered the feature matrix.
assert not any("Thermal" in c for c in X_all.columns)
assert set(X_all.columns).isdisjoint({"Thermal"})
assert X_train.index.max() < X_val.index.min()
assert X_val.index.max() < X_test.index.min()

print("Feature count:", X_all.shape[1])
print("No Thermal-derived predictors:", True)
print("Chronological split:", True)
print("Notebook pipeline is ready.")